# 02 — Cross-sectional LightGBM ranker + triple-barrier meta-model + GMM regime model

**Training contract:** Features from Notebook 01 are timestamp-safe; labels are built **separately** from future Silver bars. The ranker's relevance label is within-timestamp forward **market-excess** return quintile. The meta-model is trained only on held-out *validation* candidates, with final evaluation on a separate *test* period. A one-session embargo is used between train / validation / test; horizons are restricted to the same exchange day.

> **Evaluation is research-only:** The score and top-decile excess-return diagnostic are *not* a realistic P&L backtest. Actual trading needs exchange calendars, corporate actions, quote-based fills, spreads, borrow/fees, fills/partial fills, survivorship-free historical universe, and a portfolio simulator. No claim of beating an index is implied.

Install `lightgbm` with `%pip` if absent. We use MLflow model URIs saved to a small Delta registry table for Notebook 03. Requires Databricks Runtime ML with MLflow and scikit-learn available.

In [ ]:
%pip install -q lightgbm==4.6.0

In [ ]:
import json, math, inspect
from datetime import datetime, timezone
import numpy as np, pandas as pd
from pyspark.sql import functions as F, Window as W
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import mlflow, mlflow.sklearn

spark.conf.set("spark.sql.session.timeZone","UTC")
for name,default in [
    ("catalog","main"),("schema","stock_alpha"),
    ("experiment_path","/Shared/us_equity_alpha"),
    ("rank_horizon_bars","6"),("barrier_horizon_bars","12"),
    ("profit_target","0.006"),("stop_loss","0.004"),
    ("candidate_fraction","0.15"),
    ("train_row_limit","450000"),
    ("round_trip_cost_bps","10")
]: dbutils.widgets.text(name,default)
cfg={k:dbutils.widgets.get(k).strip() for k in ["catalog","schema","experiment_path","rank_horizon_bars","barrier_horizon_bars","profit_target","stop_loss","candidate_fraction","train_row_limit","round_trip_cost_bps"]}
B=f"`{cfg['catalog']}`.`{cfg['schema']}`"
SILVER=f"{B}.`equity_silver_5m`"; GOLD=f"{B}.`equity_features_5m`"; REGISTRY=f"{B}.`alpha_model_registry`"
H=int(cfg["rank_horizon_bars"]); BH=int(cfg["barrier_horizon_bars"])
PT=float(cfg["profit_target"]); SL=float(cfg["stop_loss"])
CAND=float(cfg["candidate_fraction"])
assert H>=1 and BH>=H and 0<PT<1 and 0<SL<1 and 0<CAND<0.5
FEATURES=["ret_1","ret_3","ret_12","ret_vol_12","rsi_14","rel_volume","clv",
          "amihud","beta_60","residual_mom_12","book_imbalance","spread_bps",
          "sector_ret_rank","sector_rvol_rank","market_resid_rank",
          "market_ret","market_vol_12","market_mom_12"]
META_FEATURES=["rank_score","ret_vol_12","rel_volume","clv","amihud","rsi_14","book_imbalance","spread_bps"]
REGIME_FEATURES=["market_ret","market_vol_12","market_mom_12"]
assert spark.catalog.tableExists(SILVER.replace('`','')) and spark.catalog.tableExists(GOLD.replace('`',''))
print('Building non-overlapping intraday forward labels on closed bars')

## Forward labels — no same-bar entry
At time *t* the final 5-minute bar has closed. Signal is generated after *t*. Hypothetical entry is at **next bar open**; horizon exit is the close of future bar `t+H`. Barriers use **future high/low only**. If both barriers cross in the same bar, we conservatively mark that bar as a stop. No label is allowed to extend past today's 16:00 ET close.

In [ ]:
s=spark.table(SILVER).select("event_time","session_date","symbol","open","high","low","close")
sw=W.partitionBy("symbol").orderBy("event_time")
labels=(s.withColumn("entry_open",F.lead("open",1).over(sw))
    .withColumn("exit_close",F.lead("close",H).over(sw))
    .withColumn("rank_end_session",F.lead("session_date",H).over(sw))
    .withColumn("barrier_end_session",F.lead("session_date",BH).over(sw))
    .withColumn("rank_end_time",F.lead("event_time",H).over(sw))
    .withColumn("barrier_end_time",F.lead("event_time",BH).over(sw)))
labels=labels.withColumn("forward_return",F.when((F.col("rank_end_session")==F.col("session_date")) &
                        ((F.col("rank_end_time").cast("long")-F.col("event_time").cast("long"))==H*300),
                        F.col("exit_close")/F.col("entry_open")-1))
# Future prices used only to compute supervised labels, NOT model input features.
for k in range(1,BH+1):
    labels=labels.withColumn(f"fh_{k}",F.lead("high",k).over(sw))                 .withColumn(f"fl_{k}",F.lead("low",k).over(sw))
profit_first=F.array_min(F.array(*[F.when(F.col(f"fh_{k}")>=F.col("entry_open")*(1+PT),F.lit(k)).otherwise(F.lit(9999)) for k in range(1,BH+1)]))
stop_first=F.array_min(F.array(*[F.when(F.col(f"fl_{k}")<=F.col("entry_open")*(1-SL),F.lit(k)).otherwise(F.lit(9999)) for k in range(1,BH+1)]))
labels=labels.withColumn("meta_label",F.when((F.col("barrier_end_session")!=F.col("session_date")) |
    ((F.col("barrier_end_time").cast("long")-F.col("event_time").cast("long"))!=BH*300),F.lit(None).cast("int"))
    .otherwise(F.when((profit_first<stop_first)&(profit_first<=BH),F.lit(1)).otherwise(F.lit(0))))
# Groupwise benchmark future return is used only for target labels (never as a feature).
bench=labels.filter(F.col("forward_return").isNotNull()).groupBy("event_time").agg(F.avg("forward_return").alias("forward_market_return"))
labels=(labels.join(bench,"event_time","left")
    .withColumn("forward_excess",F.col("forward_return")-F.col("forward_market_return"))
    .select("event_time","symbol","session_date","forward_return","forward_excess","meta_label"))

train_spark=(spark.table(GOLD).join(labels,["event_time","symbol","session_date"],"inner")
   .filter(F.col("forward_excess").isNotNull()&F.col("meta_label").isNotNull()))
# quintile relevance: LightGBM's lambdarank requires non-negative integer labels.
qw=W.partitionBy("event_time").orderBy("forward_excess","symbol")
train_spark=train_spark.withColumn("relevance",F.ntile(5).over(qw)-1)
# Use at least 15 equities per timestamp to form a meaningful cross-section.
valid_times=train_spark.groupBy("event_time").count().filter(F.col("count")>=15).select("event_time")
train_spark=train_spark.join(valid_times,"event_time","inner")
print('Labelled training candidates:',train_spark.count())

## Sample complete cross-sections (rather than randomly sampling stocks)
For large universes, keep every stock from selected timestamps, preserving the **market-wide rank group**. The cap is driver-memory protection: tune `train_row_limit` and use distributed LightGBM or Ray for larger training sets. The training fold is chronological, plus a trading-session embargo before validation and test.

In [ ]:
counts=(train_spark.groupBy("event_time").count().orderBy("event_time").toPandas())
if counts.empty: raise ValueError("No labelled data. Need >= 20+ trading sessions of closed, valid bars.")
# Deterministic, evenly spaced timestamps. Adjust stride based on approx row limit.
stride=max(1,int(math.ceil(counts['count'].sum()/int(cfg['train_row_limit']))))
selected=counts.iloc[::stride]['event_time']
selected_df=spark.createDataFrame(pd.DataFrame({'event_time':selected}))
cols=["event_time","symbol","session_date","forward_return","forward_excess","meta_label","relevance"]+FEATURES
pd_all=train_spark.join(F.broadcast(selected_df),"event_time","inner").select(*cols).toPandas()
pd_all['event_time']=pd.to_datetime(pd_all['event_time'])
pd_all['session_date']=pd.to_datetime(pd_all['session_date']).dt.date
pd_all.sort_values(['event_time','symbol'],inplace=True)
pd_all.reset_index(drop=True,inplace=True)
if len(pd_all)>int(cfg['train_row_limit'])*1.2:
    raise MemoryError('Training DataFrame too large for driver; reduce train_row_limit')
for name in FEATURES:
    pd_all[name]=pd.to_numeric(pd_all[name],errors="coerce").replace([np.inf,-np.inf],np.nan).fillna(0.0).astype('float32')
all_days=sorted(pd_all['session_date'].unique())
if len(all_days)<25: raise ValueError('Need at least 25 labelled sessions for train/val/test + embargo; backfill more history')
n=len(all_days)
train_cut=int(n*0.65); val_cut=int(n*0.82)
train_days=set(all_days[:train_cut]); val_days=set(all_days[train_cut+1:val_cut]); test_days=set(all_days[val_cut+1:])
train=pd_all[pd_all.session_date.isin(train_days)].copy()
val=pd_all[pd_all.session_date.isin(val_days)].copy()
test=pd_all[pd_all.session_date.isin(test_days)].copy()
if min(len(train),len(val),len(test))<=0: raise ValueError('No rows in one of the chronological folds')
print('Full cross-section sampling stride:',stride)
print('Train',min(train_days),max(train_days),len(train),'Val',min(val_days),max(val_days),len(val),'Test',min(test_days),max(test_days),len(test))
print('Embargo days:',all_days[train_cut],all_days[val_cut])

## Fit and diagnose the LightGBM LambdaRank model
Keep each timestamp's observations adjacent and provide the number of securities in each group. Evaluate *rank information coefficient* (cross-sectional Spearman correlation) and average top-decile forward-excess return **minus a rough cost assumption**. These are useful signal diagnostics, not compounded P&L.

In [ ]:
def ranking_groups(frame):
    return frame.groupby('event_time',sort=False).size().astype(int).to_list()
ranker=lgb.LGBMRanker(objective='lambdarank',n_estimators=220,learning_rate=0.035,
    num_leaves=31,min_child_samples=80,subsample=0.8,colsample_bytree=0.85,
    reg_lambda=3.0,verbosity=-1,random_state=42,n_jobs=-1)
ranker.fit(train[FEATURES],train['relevance'].astype(int),group=ranking_groups(train),
           eval_set=[(val[FEATURES],val['relevance'].astype(int))],eval_group=[ranking_groups(val)],
           eval_at=[5,10],callbacks=[lgb.early_stopping(25,verbose=False)])
for frame in (val,test):
    frame['rank_score']=ranker.predict(frame[FEATURES])
    frame['rank_pct']=frame.groupby('event_time')['rank_score'].rank(pct=True)

COST=float(cfg['round_trip_cost_bps'])/10000.0

def ranking_metrics(frame):
    ic=frame.groupby('event_time').apply(lambda g: g['rank_score'].corr(g['forward_excess'],method='spearman'))
    leaders=frame[frame['rank_pct']>=0.90]
    return {'mean_rank_ic':float(ic.mean()),
            'top_10pct_mean_forward_excess_after_rough_cost':float(leaders.forward_excess.mean()-COST),
            'top_10pct_signals':int(len(leaders)), 'timestamps':int(frame.event_time.nunique())}
print('VALIDATION',ranking_metrics(val))
print('TEST (untouched model-selection period)',ranking_metrics(test))

## Expanding walk-forward validation (before the reserved test set)
Fit on progressively longer **past** windows; evaluate the next short block, with one whole US trading session embargo between them. The final test period is not included in these folds. These diagnostics highlight whether the rank correlation changes by time window. This is intentionally a small research configuration: expand to many market regimes before any trading deployment.

In [ ]:
pretest_days=sorted(set(train_days)|set(val_days)|{all_days[train_cut],all_days[val_cut]})
fold_scores=[]
n_pre=len(pretest_days)
for cut in sorted(set([int(n_pre*0.55),int(n_pre*0.73)])):
    # One day immediately before each assessment is embargoed.
    fitting_dates=set(pretest_days[:max(0,cut-1)])
    evaluation_dates=set(pretest_days[cut:min(cut+max(3,int(n_pre*.13)),n_pre)])
    ft=pd_all[pd_all.session_date.isin(fitting_dates)].copy()
    fv=pd_all[pd_all.session_date.isin(evaluation_dates)].copy()
    if ft.event_time.nunique()<10 or fv.event_time.nunique()<3: continue
    fold_model=lgb.LGBMRanker(objective='lambdarank',n_estimators=100,learning_rate=0.04,
        num_leaves=31,min_child_samples=80,reg_lambda=3.0,verbosity=-1,random_state=42,n_jobs=-1)
    fold_model.fit(ft[FEATURES],ft['relevance'].astype(int),group=ranking_groups(ft))
    fv['rank_score']=fold_model.predict(fv[FEATURES])
    fv['rank_pct']=fv.groupby('event_time')['rank_score'].rank(pct=True)
    m=ranking_metrics(fv)
    fold_scores.append({'train_last_day':str(max(fitting_dates)),'eval_first_day':str(min(evaluation_dates)),**m})
print('Expanding walk-forward folds (not including final test):')
for f in fold_scores: print(f)

## Triple-barrier meta-model on validation candidates only
Avoid fitting a meta-model on ranker predictions from its own training rows: that creates stacking leakage. The validation period is used for meta-model fitting, and the test period remains untouched until final evaluation. The 1/0 label only means the profit barrier was hit before stop/time within the chosen horizon; it is **not a calibrated chance of profit** without further reliability tests.

In [ ]:
val_candidates=val[val['rank_pct'] >= 1-CAND].copy()
test_candidates=test[test['rank_pct'] >= 1-CAND].copy()
for frame in (val_candidates,test_candidates):
    for c in META_FEATURES:
        frame[c]=pd.to_numeric(frame[c],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0)
meta=None
if len(val_candidates)>=80 and val_candidates['meta_label'].nunique()==2:
    meta=RandomForestClassifier(n_estimators=180,min_samples_leaf=15,max_features='sqrt',
                                class_weight='balanced_subsample',random_state=42,n_jobs=-1)
    meta.fit(val_candidates[META_FEATURES],val_candidates['meta_label'].astype(int))
    test_candidates['meta_probability']=meta.predict_proba(test_candidates[META_FEATURES])[:,list(meta.classes_).index(1)]
    test_candidates['paper_meta_accept']=test_candidates['meta_probability']>=0.55
    accepted=test_candidates[test_candidates['paper_meta_accept']]
    print('Meta validation labels:',val_candidates['meta_label'].value_counts().to_dict())
    print('TEST meta accepted:',len(accepted),'of',len(test_candidates))
    print('TEST meta observed positive-barrier frequency (accepted only):',float(accepted.meta_label.mean()) if len(accepted) else None)
else:
    print('Meta disabled: insufficient or single-class validation candidates. Ranker still available.')

## Market-regime GMM
This unsupervised model clusters current, observable market-average return, 12-bar rolling market volatility and 12-bar momentum; it sees **only the training fold** during fit. Regimes are ordered by **in-sample volatility** rather than guessed bull/bear names. Notebook 03 can block new paper buys in `HIGH_VOL` and reduce size in `MID_VOL`.


In [ ]:
regime_df=(spark.table(GOLD)
    .select('event_time','session_date',*REGIME_FEATURES).dropDuplicates(['event_time'])
    .filter(F.col('market_vol_12').isNotNull())
    .toPandas())
regime_df['session_date']=pd.to_datetime(regime_df['session_date']).dt.date
reg_train=regime_df[regime_df['session_date'].isin(train_days)].copy()
for c in REGIME_FEATURES: reg_train[c]=pd.to_numeric(reg_train[c],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0)
if len(reg_train)<50: raise ValueError('Not enough market timestamps to train regime model')
regime_model=make_pipeline(StandardScaler(),GaussianMixture(n_components=3,covariance_type='full',reg_covar=1e-5,n_init=5,random_state=42))
regime_model.fit(reg_train[REGIME_FEATURES])
reg_train['cluster']=regime_model.predict(reg_train[REGIME_FEATURES])
vols=reg_train.groupby('cluster')['market_vol_12'].mean().sort_values()
REGIME_NAMES={int(cluster):label for cluster,label in zip(vols.index,['LOW_VOL','MID_VOL','HIGH_VOL'])}
print('Regime mapping:',REGIME_NAMES,'in-sample average vol:',vols.to_dict())

## Log reproducible models in MLflow; write model URIs to a Delta registry
Model URIs are stored *exactly as MLflow reports them* (compatible with MLflow 3's `models:/m-...` URIs). Notebook 03 always loads the latest successful registry entry, along with exact ordered feature lists and thresholds. Treat the registry as a deployment staging area; review metrics and approve a production release separately.

In [ ]:
mlflow.set_experiment(cfg['experiment_path'])
with mlflow.start_run(run_name='market_cross_sectional_rank_meta_gmm') as run:
    mlflow.log_params({
        'rank_horizon_bars':H,'barrier_horizon_bars':BH,'profit_target':PT,'stop_loss':SL,
        'candidate_fraction':CAND,'cost_bps':float(cfg['round_trip_cost_bps']),
        'sample_stride':stride,'train_rows':len(train),'val_rows':len(val),'test_rows':len(test),
        'feature_count':len(FEATURES),'embargo_trading_sessions':1})
    metrics=ranking_metrics(test)
    mlflow.log_metrics({k:float(v) for k,v in metrics.items()})
    def save_model(obj,name):
        args={'name':name} if 'name' in inspect.signature(mlflow.sklearn.log_model).parameters else {'artifact_path':name}
        return mlflow.sklearn.log_model(sk_model=obj,**args).model_uri
    ranker_uri=save_model(ranker,'cross_sectional_ranker')
    meta_uri=save_model(meta,'triple_barrier_meta') if meta is not None else ''
    regime_uri=save_model(regime_model,'market_regime_gmm')
    mlflow.log_dict({'features':FEATURES,'meta_features':META_FEATURES,
                     'regime_features':REGIME_FEATURES,'regime_names':REGIME_NAMES},'training_config.json')
    run_id=run.info.run_id

record=[(datetime.now(timezone.utc),run_id,ranker_uri,meta_uri,regime_uri,
         json.dumps(FEATURES),json.dumps(META_FEATURES),json.dumps(REGIME_FEATURES),
         json.dumps(REGIME_NAMES),float(CAND),float(PT),float(SL),int(H),int(BH),
         float(metrics['mean_rank_ic']))]
columns=['trained_at','run_id','ranker_uri','meta_uri','regime_uri','features_json','meta_features_json',
         'regime_features_json','regime_names_json','candidate_fraction','profit_target','stop_loss',
         'rank_horizon_bars','barrier_horizon_bars','test_mean_rank_ic']
reg=spark.createDataFrame(record,schema=columns)
reg.write.format('delta').mode('append').saveAsTable(REGISTRY)
print('Run:',run_id)
print('Ranker URI:',ranker_uri)
print('Meta URI:',meta_uri or '(disabled)')
print('Regime URI:',regime_uri)
display(spark.table(REGISTRY).orderBy(F.desc('trained_at')).limit(3))